<a href="https://colab.research.google.com/github/harshvarudkar/test/blob/master/Demo0507.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install -U langchain-core langchain-community langchain-openai langgraph pydantic

In [17]:
import os
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
llm = ChatOpenAI(model="gpt-4o-mini")

In [20]:
# Basic LLM Call
response = llm.invoke("What is 100 C into fahrenheit?")
print(response.content)

To convert degrees Celsius (C) to degrees Fahrenheit (F), you can use the formula:

\[
F = \frac{9}{5}C + 32
\]

For 100 degrees Celsius:

\[
F = \frac{9}{5} \times 100 + 32
\]
\[
F = 180 + 32
\]
\[
F = 212
\]

Therefore, 100 degrees Celsius is equal to 212 degrees Fahrenheit.


In [21]:
# PromptTemplate
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Convert {value1} {unit1} to {unit2}"
)

final_prompt = prompt.invoke({"value1": 100, "unit1": "meter", "unit2" : "feet"})
response  = llm.invoke(final_prompt)
print(response.content)

To convert meters to feet, you can use the conversion factor: 1 meter is approximately equal to 3.28084 feet.

So, to convert 100 meters to feet:

100 meters × 3.28084 feet/meter ≈ 328.084 feet.

Therefore, 100 meters is approximately 328.08 feet.


In [24]:
# PromptTemplate Assign a role
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ('system', 'You are a temperature convertor which will just return number.'),
        ('user', "What is {temp} Celsius  into fahrenheit?")
    ]
)

chain_input = prompt.invoke({"temp": "100"})
response = llm.invoke(chain_input)
print(response.content)



212


In [29]:
# Code Generated by Sidekick is for learning and experimentation purposes only.
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a temperature converter and must return only the number."),
    ("user", "What is {temperature} C into fahrenheit?")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

parser = StrOutputParser()

chain = prompt | llm | parser

result = chain.invoke({"temperature": "100"})
print(result)  # likely: 212


212


In [32]:
# Code Generated by Sidekick is for learning and experimentation purposes only.
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a temperature converter and must return Return valid JSON only."),
    ("user", "What is {temperature} C into fahrenheit?")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

parser = JsonOutputParser()

chain = prompt | llm | parser

result = chain.invoke({"temperature": "100"})
print(result)  # likely: 212

{'celsius': 100, 'fahrenheit': 212}


In [34]:
response = llm.invoke("What is temperature of Pune now")
print(response.content)

I'm unable to provide real-time data, including current temperatures. To find the current temperature in Pune, I recommend checking a reliable weather website or using a weather app.


In [66]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Return the current weather for a given city."""
    return f"The weather in {city} is sunny."

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[get_weather],
    system_prompt="You are a helpful assistant."
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in Dallas?"}]}
)

print(result["messages"][-1].content)



The weather in Dallas is sunny.


In [65]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

@tool
def get_current_year() -> int:
    """Return the current year."""
    return 2026

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[multiply, get_current_year],
    system_prompt="You are a helpful AI agent. Use tools when needed."
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is 7 times 8, and what year is it?"}
    ]
})

print(result["messages"][-1].content)


7 times 8 is 56, and the current year is 2026.


In [67]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

@tool
def get_current_year() -> int:
    """Return the current year."""
    return 2026

def run_tools(inputs: dict) -> dict:
    question = inputs["question"]
    product = multiply.invoke({"a": 7, "b": 8})
    year = get_current_year.invoke({})
    return {
        "question": question,
        "product": product,
        "year": year,
    }

prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant.\n"
    "User question: {question}\n"
    "Computed result for 7 * 8: {product}\n"
    "Current year: {year}\n"
    "Answer clearly in one sentence."
)

model = ChatOpenAI(model="gpt-4.1-mini")
parser = StrOutputParser()

chain = RunnableLambda(run_tools) | prompt | model | parser

result = chain.invoke({
    "question": "What is 7 times 8, and what year is it?"
})

print(result)


7 times 8 is 56, and the current year is 2026.
